In [1]:
from pathlib import Path
import json

config = json.loads(Path("mec_config.json").read_text(encoding="utf-8"))
required_files = config["required_files"]
missing_files = [name for name in required_files if not Path(name).exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")
print("Local notebook execution ready")

Local notebook execution ready


In [2]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.columns)
print(df['label'].unique())


Index(['timestamp', 'vehicle_id', 'label', 'speed_kmh', 'fuel_liters',
       'fuel_pct', 'battery_percent', 'engine_temp_c', 'payload_tons',
       'grade_pct', 'latitude', 'longitude', 'rsrp_dbm', 'sinr_db', 'rtt_ms',
       'packet_loss_percent', 'ul_mbps', 'dl_mbps', 'handover', 'in_tunnel',
       'waypoint_idx', 'broken', 'flat_tyre', 'ambient_temp_c', 'overloaded',
       'low_fuel_warning'],
      dtype='object')
['syn_flood' 'udp_flood' 'slowloris_attack' 'icmp_flood'
 'network_degradation_attack' 'normal']


In [3]:
df['attack_label'] = df['label']

In [4]:
def get_protocol(label):
    label = str(label).lower()
    for rule in config["protocol_rules"]:
        if any(keyword in label for keyword in rule["keywords"]):
            return rule["protocol"]
    return "unknown"

df['protocol'] = df['label'].apply(get_protocol)

In [5]:
protocol_df = df[df['protocol'] != 'unknown'].copy()

In [6]:
def get_quality(row):
    quality_rules = config["quality_rules"]
    if row['packet_loss_percent'] < quality_rules["good"]["packet_loss_percent_lt"] and row['rtt_ms'] < quality_rules["good"]["rtt_ms_lt"]:
        return "good"
    if row['packet_loss_percent'] < quality_rules["normal"]["packet_loss_percent_lt"] and row['rtt_ms'] < quality_rules["normal"]["rtt_ms_lt"]:
        return "normal"
    return quality_rules["fallback"]

df['quality'] = df.apply(get_quality, axis=1)

In [7]:
from sklearn.preprocessing import StandardScaler

feature_columns = config["feature_columns"]
X = df[feature_columns]
X_protocol = protocol_df[feature_columns]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_protocol_scaled = scaler.transform(X_protocol)

In [8]:
from sklearn.model_selection import train_test_split

training = config["training"]

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_scaled, df['attack_label'], test_size=training["test_size"], random_state=training["random_state"], stratify=df['attack_label']
)

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_protocol_scaled, protocol_df['protocol'], test_size=training["test_size"], random_state=training["random_state"], stratify=protocol_df['protocol']
)

X_train_q, X_test_q, y_train_q, y_test_q = train_test_split(
    X_scaled, df['quality'], test_size=training["test_size"], random_state=training["random_state"], stratify=df['quality']
)

In [9]:
import numpy as np

X_test_a += np.random.normal(0, 0.05, X_test_a.shape)
X_test_p += np.random.normal(0, 0.05, X_test_p.shape)
X_test_q += np.random.normal(0, 0.05, X_test_q.shape)

In [10]:
from sklearn.ensemble import RandomForestClassifier

model_attack = RandomForestClassifier(**config["training"]["attack_model"])
model_protocol = RandomForestClassifier(**config["training"]["protocol_model"])
model_quality = RandomForestClassifier(**config["training"]["quality_model"])

model_attack.fit(X_train_a, y_train_a)
model_protocol.fit(X_train_p, y_train_p)
model_quality.fit(X_train_q, y_train_q)

,n_estimators,50
,criterion,'gini'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
from sklearn.metrics import accuracy_score

# Attack
pred_a = model_attack.predict(X_test_a)
acc_a = accuracy_score(y_test_a, pred_a)

# Protocol
pred_p = model_protocol.predict(X_test_p)
acc_p = accuracy_score(y_test_p, pred_p)

# Quality
pred_q = model_quality.predict(X_test_q)
acc_q = accuracy_score(y_test_q, pred_q)

print("Attack Accuracy:", round(acc_a * 100, 2), "%")
print("Protocol Accuracy:", round(acc_p * 100, 2), "%")
print("Quality Accuracy:", round(acc_q * 100, 2), "%")

Attack Accuracy: 89.29 %
Protocol Accuracy: 85.28 %
Quality Accuracy: 99.15 %


In [12]:
import joblib

joblib.dump(model_attack, config["model_files"]["attack"])
joblib.dump(model_protocol, config["model_files"]["protocol"])
joblib.dump(model_quality, config["model_files"]["quality"])
joblib.dump(scaler, config["model_files"]["scaler"])
print({name: Path(path).exists() for name, path in config["model_files"].items()})

{'scaler': True, 'attack': True, 'protocol': True, 'quality': True}


In [13]:
def predict(data):
    data_df = pd.DataFrame([data], columns=feature_columns)
    data_scaled = scaler.transform(data_df)

    attack = model_attack.predict(data_scaled)[0]
    protocol = model_protocol.predict(data_scaled)[0]
    quality = model_quality.predict(data_scaled)[0]

    return {
        "attack_type": attack,
        "protocol": protocol,
        "traffic_quality": quality,
        "is_attack": attack != "normal"
    }


In [14]:
sample = [30, 1, 50, 20, -60, 30]

result = predict(sample)
print("Sample Prediction:", result)

Sample Prediction: {'attack_type': 'normal', 'protocol': 'tcp', 'traffic_quality': 'good', 'is_attack': False}
